## Extração de Licitações (dadosabertos.compras.gov.br)

Extrai licitações de PE via API de Dados Abertos do Compras.gov.br.

**Estratégia confirmada via Swagger:**
- `uasg` é **opcional** — consultar sem filtro de UASG e filtrar PE localmente
- `data_publicacao_inicial` e `data_publicacao_final` são **obrigatórios** (formato `YYYY-MM-DD`)
- Iteração mês a mês: ~60 chamadas totais (vs 6.000 se fosse por UASG × ano)

**Saídas:** `data/raw/licitacoes_legado_pe.csv`, `data/raw/contratacoes14133_pe.csv`

In [21]:
import urllib.request
import urllib.parse
import json
import pandas as pd
import calendar
import time
import os

os.makedirs("data/raw", exist_ok=True)
BASE = "https://dadosabertos.compras.gov.br"

uasg_pe = pd.read_csv("data/raw/uasg_pe.csv")
uasgs_pe = set(uasg_pe["codigoUasg"].astype(int).tolist())
print(f"UASGs de PE carregadas: {len(uasgs_pe)}")

UASGs de PE carregadas: 1197


### 1. Debug — Validar endpoint sem filtro de UASG

In [ ]:
url = BASE + "/modulo-legado/1_consultarLicitacao?" + urllib.parse.urlencode({
    "data_publicacao_inicial": "2024-01-01",
    "data_publicacao_final":   "2024-01-31",
    "pagina":        1,
    "tamanhoPagina": 10
})
print("URL:", url)

with urllib.request.urlopen(url, timeout=30) as resp:
    r = json.loads(resp.read().decode())

registros = r.get("resultado", r) if isinstance(r, dict) else r
print(f"Keys: {list(r.keys()) if isinstance(r, dict) else '(lista)'}")
print(f"Registros na página: {len(registros)}")

de_pe = [rec for rec in registros if int(rec.get("uasg", 0)) in uasgs_pe]
print(f"Desses, de PE: {len(de_pe)}")

if registros:
    print("\nCampos disponíveis:")
    for k, v in registros[0].items():
        print(f"  {k}: {str(v)[:120]}")

URL: https://dadosabertos.compras.gov.br/modulo-legado/1_consultarLicitacao?data_publicacao_inicial=2024-01-01&data_publicacao_final=2024-01-31&pagina=1&tamanhoPagina=10
Keys: ['resultado', 'totalRegistros', 'totalPaginas', 'paginasRestantes']
Registros na página: 10
Desses, de PE: 3

Campos disponíveis:
  id_compra: 16019405000152023
  identificador: 16019405152023
  numero_processo: 64318056248202339
  uasg: 160194
  modalidade: 5
  nome_modalidade: PREGÃO
  numero_aviso: 152023
  situacao_aviso: Publicado
  tipo_pregao: eletronico
  tipo_recurso: Nacional
  nome_responsavel: ALEXANDRE DA SILVA GALDINO
  funcao_responsavel: Ordenador de Despesas
  numero_itens: 33
  valor_estimado_total: None
  valor_homologado_total: 107.38
  informacoes_gerais: None
  objeto: Pregão Eletrônico Contratação de emp resa, pelo sistema de registro de preços, para prestação dos serviços de transporte
  endereco_entrega_edital: Av. Visconde de Sao Leopoldo, 198 - Engenho do Meio - Várzea/PE
  codigo_munic

### 2. Extração completa — Licitações legado (2020–2024)

Itera mês a mês para todo o Brasil e filtra PE localmente pelo `set` de UASGs.
Checkpoint salvo ao final de cada ano.

In [ ]:
TAM_PAGINA = 500
RETRY_ESPERAS = [5, 15, 30]  

def get_licitacoes_mes(ano, mes, uasgs_filtro=None):
    ultimo_dia = calendar.monthrange(ano, mes)[1]
    data_ini = f"{ano}-{mes:02d}-01"
    data_fim = f"{ano}-{mes:02d}-{ultimo_dia:02d}"
    resultados = []
    pagina = 1
    while True:
        url = BASE + "/modulo-legado/1_consultarLicitacao?" + urllib.parse.urlencode({
            "data_publicacao_inicial": data_ini,
            "data_publicacao_final":   data_fim,
            "pagina":        pagina,
            "tamanhoPagina": TAM_PAGINA
        })
        page = None
        for tentativa, espera in enumerate(RETRY_ESPERAS + [None], start=1):
            try:
                with urllib.request.urlopen(url, timeout=30) as resp:
                    r = json.loads(resp.read().decode())
                page = r.get("resultado", r) if isinstance(r, dict) else r
                break 
            except Exception as e:
                if espera is None:
                    print(f"    FALHA DEFINITIVA {ano}-{mes:02d} p{pagina}: {e}")
                else:
                    print(f"    Tentativa {tentativa} falhou ({e}). Aguardando {espera}s...")
                    time.sleep(espera)
        if page is None:
            break 
        if not page:
            break
        if uasgs_filtro:
            page = [rec for rec in page if int(rec.get("uasg", 0)) in uasgs_filtro]
        resultados.extend(page)
        if len(page) < TAM_PAGINA:
            break
        pagina += 1
        time.sleep(0.3)
    return resultados

MESES_FALTANTES = [
    (2020, 8), (2020, 11),
    (2021, 7), (2021, 8), (2021, 10),
    (2022, 9),
    (2023, 3),
    (2024, 1),
]

df_legado = pd.read_csv("data/raw/licitacoes_legado_pe.csv")
recuperados = []
for ano, mes in MESES_FALTANTES:
    lics = get_licitacoes_mes(ano, mes, uasgs_filtro=uasgs_pe)
    for l in lics:
        l["fonte"] = "legado"
    recuperados.extend(lics)
    print(f"  {ano}-{mes:02d}: {len(lics):4d} licitações recuperadas")
    time.sleep(1)

if recuperados:
    df_rec = pd.DataFrame(recuperados)
    df_legado = pd.concat([df_legado, df_rec], ignore_index=True)
    df_legado = df_legado.drop_duplicates(subset=["id_compra"])
    df_legado.to_csv("data/raw/licitacoes_legado_pe.csv", index=False)
    print(f"\nTotal após recuperação: {len(df_legado)} licitações")
else:
    print("Nenhum registro recuperado.")
df_legado.head(3)

  2020-08:   16 licitações recuperadas
  2020-11:   13 licitações recuperadas
  2021-07:   21 licitações recuperadas
  2021-08:   26 licitações recuperadas
  2021-10:   13 licitações recuperadas
  2022-09:   19 licitações recuperadas
  2023-03:   34 licitações recuperadas
  2024-01:   21 licitações recuperadas

Total após recuperação: 1328 licitações


,id_compra,identificador,numero_processo,uasg,modalidade,nome_modalidade,numero_aviso,situacao_aviso,tipo_pregao,tipo_recurso,...,objeto,endereco_entrega_edital,codigo_municipio_uasg,data_abertura_proposta,data_entrega_edital,data_entrega_proposta,data_publicacao,dt_alteracao,pertence14133,fonte
0,92617105000012020,9261710512020,25800.003387/2019,926171,5,PREGÃO,12020,Publicado,eletronico,Nacional,...,Pregão Eletrônico Contratação de ser viços de ...,"Rua Professor Aloísio Pessoa de Araújo, Nº 75,...",25313,2020-01-14,2020-01-02,2020-01-02,2020-01-02,2020-01-14T17:01:46,False,legado
1,8000605000412019,8000605412019,19.776/19,80006,5,PREGÃO,412019,Publicado,eletronico,Nacional,...,Pregão Eletrônico Aquisição de papel A4 branco.,"Cais do Apolo Nº 739, Bairro do Recife, Recife...",25313,2020-01-15,2020-01-02,2020-01-02,2020-01-02,2020-03-04T12:59:15,False,legado
2,92617105000022020,9261710522020,25800.005214/2019,926171,5,PREGÃO,22020,Publicado,eletronico,Nacional,...,Pregão Eletrônico Contratação de emp resa espe...,"Rua Professor Aloísio Pessoa de Araújo, Nº 75,...",25313,2020-01-15,2020-01-03,2020-01-03,2020-01-03,2020-02-10T15:51:38,False,legado


### 3. Debug — Contratações 14133: inspecionar estrutura real da resposta

O módulo 14133 tem parâmetros **diferentes** do legado (camelCase + `codigoModalidade` obrigatório).
Usa `unidadeOrgaoUfSigla=PE` para filtrar diretamente no servidor.

In [ ]:
url_dbg = BASE + "/modulo-contratacoes/1_consultarContratacoes_PNCP_14133?" + urllib.parse.urlencode({
    "dataPublicacaoPncpInicial": "2024-01-01",
    "dataPublicacaoPncpFinal":   "2024-01-31",
    "codigoModalidade":          6,          
    "unidadeOrgaoUfSigla":       "PE",       
    "pagina":        1,
})
print("URL:", url_dbg)

with urllib.request.urlopen(url_dbg, timeout=30) as resp:
    r_dbg = json.loads(resp.read().decode())

recs_dbg = r_dbg.get("resultado", r_dbg) if isinstance(r_dbg, dict) else r_dbg
print(f"Keys do response: {list(r_dbg.keys()) if isinstance(r_dbg, dict) else '(lista)'}")
print(f"Registros: {len(recs_dbg)}")
if recs_dbg:
    print("\nCampos disponíveis:")
    for k, v in recs_dbg[0].items():
        print(f"  {k}: {str(v)[:120]}")

URL: https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133?dataPublicacaoPncpInicial=2024-01-01&dataPublicacaoPncpFinal=2024-01-31&codigoModalidade=6&unidadeOrgaoUfSigla=PE&pagina=1
Keys do response: ['resultado', 'totalRegistros', 'totalPaginas', 'paginasRestantes']
Registros: 10

Campos disponíveis:
  idCompra: 15316506900002024
  numeroControlePNCP: 24416174000106-1-000001/2024
  anoCompraPncp: 2024
  sequencialCompraPncp: 1
  orgaoEntidadeCnpj: 24416174000106
  orgaoSubrogadoCnpj: None
  codigoOrgao: 75030
  orgaoEntidadeRazaoSocial: UNIVERSIDADE FEDERAL RURAL DE PERNAMBUCO
  orgaoSubrogadoRazaoSocial: None
  orgaoEntidadeEsferaId: F
  orgaoSubrogadoEsferaId: None
  orgaoEntidadePoderId: E
  orgaoSubrogadoPoderId: None
  unidadeOrgaoCodigoUnidade: 153165
  unidadeSubrogadaCodigoUnidade: None
  unidadeOrgaoNomeUnidade: UNIVERSIDADE FEDERAL RURAL DE PERNAMBUCO
  unidadeSubrogadaNomeUnidade: None
  unidadeOrgaoUfSigla: PE
  unidadeSubrogadaUfSigla:

### 4. Extração completa — Contratações 14133 (2023–2024)

Itera sobre todas as modalidades × mês × ano, filtrando por `unidadeOrgaoUfSigla=PE` no servidor.

**Modalidades Lei 14.133:**
- 1 Leilão Eletrônico | 2 Diálogo Competitivo | 3 Concurso
- 4 Concorrência Eletrônica | 5 Concorrência Presencial
- 6 Pregão Eletrônico | 7 Pregão Presencial
- 8 Dispensa | 9 Inexigibilidade | 10 Manifestação de Interesse
- 11 Pré-qualificação | 12 Credenciamento | 13 Leilão Presencial

In [ ]:
MODALIDADES_14133 = list(range(1, 14))  
RETRY_ESPERAS = [5, 15, 30]

def get_c14133_modalidade_mes(modalidade, ano, mes):
    ultimo_dia = calendar.monthrange(ano, mes)[1]
    data_ini = f"{ano}-{mes:02d}-01"
    data_fim = f"{ano}-{mes:02d}-{ultimo_dia:02d}"
    resultados = []
    pagina = 1
    while True:
        url = BASE + "/modulo-contratacoes/1_consultarContratacoes_PNCP_14133?" + urllib.parse.urlencode({
            "dataPublicacaoPncpInicial": data_ini,
            "dataPublicacaoPncpFinal":   data_fim,
            "codigoModalidade":          modalidade,
            "unidadeOrgaoUfSigla":       "PE",
            "pagina":        pagina,
            "tamanhoPagina": TAM_PAGINA
        })
        page = None
        for tentativa, espera in enumerate(RETRY_ESPERAS + [None], start=1):
            try:
                with urllib.request.urlopen(url, timeout=30) as resp:
                    r = json.loads(resp.read().decode())
                page = r.get("resultado", r) if isinstance(r, dict) else r
                break
            except Exception as e:
                if espera is None:
                    print(f"    FALHA {ano}-{mes:02d} mod{modalidade} p{pagina}: {e}")
                else:
                    time.sleep(espera)
        if page is None:
            break
        if not page:
            break
        resultados.extend(page)
        if len(page) < TAM_PAGINA:
            break
        pagina += 1
        time.sleep(0.3)
    return resultados


todas_c14 = []
for ano in [2023, 2024]:
    for mes in range(1, 13):
        total_mes = 0
        for modalidade in MODALIDADES_14133:
            cs = get_c14133_modalidade_mes(modalidade, ano, mes)
            for c in cs:
                c["fonte"] = "14133"
                c["codigoModalidade"] = modalidade
            todas_c14.extend(cs)
            total_mes += len(cs)
            time.sleep(0.2)
        print(f"  {ano}-{mes:02d}: {total_mes:4d} contrataçoes PE  (total acumulado: {len(todas_c14)})")
    pd.DataFrame(todas_c14).to_csv(f"data/raw/contratacoes14133_pe_ate{ano}.csv", index=False)
    print(f">>> Checkpoint {ano}: {len(todas_c14)} contrataçoes")

df_c14 = pd.DataFrame(todas_c14)
df_c14.to_csv("data/raw/contratacoes14133_pe.csv", index=False)
print(f"\n{len(df_c14)} contrataçoes 14133 salvas em data/raw/contratacoes14133_pe.csv")
df_c14.head(3)

  2023-01:   61 contrataçoes PE  (total acumulado: 61)
  2023-02:   74 contrataçoes PE  (total acumulado: 135)
  2023-03:  162 contrataçoes PE  (total acumulado: 297)
  2023-04:  175 contrataçoes PE  (total acumulado: 472)
  2023-05:  300 contrataçoes PE  (total acumulado: 772)
  2023-06:  288 contrataçoes PE  (total acumulado: 1060)
  2023-07:  296 contrataçoes PE  (total acumulado: 1356)
  2023-08:  358 contrataçoes PE  (total acumulado: 1714)
  2023-09:  333 contrataçoes PE  (total acumulado: 2047)
  2023-10:  356 contrataçoes PE  (total acumulado: 2403)
  2023-11:  347 contrataçoes PE  (total acumulado: 2750)
  2023-12:  303 contrataçoes PE  (total acumulado: 3053)
>>> Checkpoint 2023: 3053 contrataçoes
  2024-01:  187 contrataçoes PE  (total acumulado: 3240)
  2024-02:  310 contrataçoes PE  (total acumulado: 3550)
  2024-03:  483 contrataçoes PE  (total acumulado: 4033)
  2024-04: 1362 contrataçoes PE  (total acumulado: 5395)
  2024-05:  732 contrataçoes PE  (total acumulado: 6127

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,fonte
0,38919505000012023,09791450000114-1-000001/2023,2023,1,09791450000114,None,4243,CONSELHO REGIONAL DE MEDICINA VETERINARIA DE P...,None,F,...,Aberto,94740.6,92341.20,2023-01-12T07:01:54,2023-01-12T07:01:54,2023-01-12T07:01:54,2023-01-12T08:00:00,2023-01-26T09:00:00,False,14133
1,15518005000572022,15126437000143-1-000842/2022,2022,842,15126437000143,None,95159,EMPRESA BRASILEIRA DE SERVIÇOS HOSPITALARES,None,F,...,Aberto,0.0,386717.83,2023-01-13T11:43:40,2023-01-13T11:43:40,2023-01-13T11:43:40,2023-01-13T08:00:00,2023-01-25T09:00:00,False,14133
2,15518005000022023,15126437000143-1-000043/2023,2023,43,15126437000143,None,95159,EMPRESA BRASILEIRA DE SERVIÇOS HOSPITALARES,None,F,...,Aberto,0.0,263729.60,2023-01-20T07:00:56,2023-01-20T07:00:56,2023-01-20T07:00:56,2023-01-20T08:00:00,2023-02-01T09:00:00,False,14133
